In [64]:
from dotenv import load_dotenv
load_dotenv()

True

In [65]:
from openai import OpenAI
openai_client = OpenAI()

from ingest import *

In [66]:
documents = load_faq_data()
index = build_index(documents)

In [67]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I still join?"},]

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [68]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [69]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# First CALL to the LLM

In [70]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [71]:
print(response.output[0])

ResponseFunctionToolCall(arguments='{"query":"Can I still join course late enrollment discovered the course join now"}', call_id='call_LkK3NeLvlvLj2NHyrHm96Dk3', name='search', type='function_call', id='fc_02367c2bebbc1aab006a390ff347e8819dbd9869b68e27b2cf', namespace=None, status='completed')


In [72]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [73]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certifi

In [74]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [75]:
messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I still join?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I still join course late enrollment discovered the course join now"}', call_id='call_LkK3NeLvlvLj2NHyrHm96Dk3', name='search', type='function_call', id='fc_02367c2bebbc1aab006a390ff347e8819dbd9869b68e27b2cf', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_LkK3NeLvlvLj2NHyrHm96Dk3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n

Second CALL

In [76]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [77]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(813, 29)

In [78]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [79]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [80]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }


In [81]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [82]:
messages.extend(response.output)
has_function_calls = False

In [83]:
print(messages)

[{'role': 'developer', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join enrollment late join FAQ"}', call_id='call_AbWYfdUs8AqeNsHrkmUCs5q0', name='search', type='function_call', id='fc_0497b40c37951dc3006a390ff5e1e8819caecd501d4bca5419', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"new student join course registration enroll after course start FAQ"}', call_id=

In [84]:
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course discovered course can I join enrollment late join FAQ"}
function_call: search {"query":"new student join course registration enroll after course start FAQ"}


In [85]:
(messages)

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join the course discovered course can I join enrollment late join FAQ"}', call_id='call_AbWYfdUs8AqeNsHrkmUCs5q0', name='search', type='function_call', id='fc_0497b40c37951dc3006a390ff5e1e8819caecd501d4bca5419', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student join course registration enroll after course start FAQ"}', cal

In [86]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]



it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


function_call: search {"query":"join course discovered course can I join enrollment registration late join"}
function_call: search {"query":"course join after start can I enroll late registration FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

A couple of notes:
- You can start anytime and work through the materials at your own pace.
- If you want a certificate, you need to submit your project while submissions are still open.
- If you’re joining after the live cohort period, self-paced participation is still possible, but certificates aren’t awarded in self-paced mode.

If you want, I can also help you figure out the best way to start from where you are now. Are there other areas you want to explore?


In [87]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [89]:
agent_loop(instructions, "Where is the ropesitory of the LLM courser?")

iteration #1...
function_call: search {"query":"LLM course repository repository course GitHub"}
function_call: search {"query":"LLM courser ropesitory repository"}
function_call: search {"query":"course FAQ LLM repository code materials"}
iteration #2...
ASSISTANT:
The LLM Zoomcamp repository is here: https://github.com/DataTalksClub/llm-zoomcamp

You can also use the course docs here: https://datatalks.club/docs/courses/llm-zoomcamp/

If you meant a specific lesson or homework repo, tell me which week/module and I can help find that too. Is there anything else you want to explore?


'The LLM Zoomcamp repository is here: https://github.com/DataTalksClub/llm-zoomcamp\n\nYou can also use the course docs here: https://datatalks.club/docs/courses/llm-zoomcamp/\n\nIf you meant a specific lesson or homework repo, tell me which week/module and I can help find that too. Is there anything else you want to explore?'